# 🧪 Notebook 01: EDA & RFM Feature Engineering
### Project: Professional Customer Segmentation & CLV Prediction

This notebook focuses on transforming raw transactional data (~1M rows) into a structured RFM matrix and generating supervised targets for Customer Lifetime Value (CLV).

In [1]:
import pandas as pd
import numpy as np
import os
import datetime as dt

# Determine base path relative to this notebook
BASE_DIR = os.path.dirname(os.getcwd()) if "notebooks" in os.getcwd() else os.getcwd()
DATA_DIR = os.path.join(BASE_DIR, 'data', 'raw')
PROCESSED_DIR = os.path.join(BASE_DIR, 'data', 'processed')
MODELS_DIR = os.path.join(BASE_DIR, 'models')
CHARTS_DIR = os.path.join(BASE_DIR, 'charts')

# Create directories if they don't exist
os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(CHARTS_DIR, exist_ok=True)

## 1. Load and Initial Clean
We drop records missing Customer ID and handle cancelled orders (Quantity < 0).
*Note: Dataset columns are ['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country']*.

In [2]:
csv_path = os.path.join(DATA_DIR, 'online_retail_II.csv')
print(f"Loading dataset from {csv_path}...")

if not os.path.exists(csv_path):
    raise FileNotFoundError(f"Dataset not found at {csv_path}")

df = pd.read_csv(csv_path, encoding='unicode_escape')

# Pre-cleaning
df.dropna(subset=['Customer ID'], inplace=True)
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
df['TotalPrice'] = df['Quantity'] * df['Price']

# Filter out returns for RFM but keep them as background if needed (here we simplify to positive only)
df = df[df['Quantity'] > 0]
df = df[df['Price'] > 0]

print(f"Cleaned shape: {df.shape}")

Loading dataset from /home/ashutosh/Desktop/NO-AI-USE/ml_projects/03_customer_segmentation_clv/data/raw/online_retail_II.csv...


Cleaned shape: (805549, 9)


## 2. Define Time-Split for CLV
To predict future value, we use
- **Calibration Period**: Dec 2009 to June 2011 (to build features)
- **Observation Period**: July 2011 to Dec 2011 (to calculate the target: future spend)

In [3]:
split_date = pd.Timestamp("2011-06-01")

df_calib = df[df['InvoiceDate'] < split_date]
df_obs = df[df['InvoiceDate'] >= split_date]

print(f"Calibration records: {len(df_calib)}")
print(f"Observation records: {len(df_obs)}")

Calibration records: 553114
Observation records: 252435


## 3. Engineer RFM Features (Calibration)
- **Recency**: Days since last purchase (relative to split date)
- **Frequency**: Count of unique invoices
- **Monetary**: Sum of TotalPrice

In [4]:
snapshot_date = split_date

rfm = df_calib.groupby('Customer ID').agg({
    'InvoiceDate': lambda x: (snapshot_date - x.max()).days,
    'Invoice': 'nunique',
    'TotalPrice': 'sum'
})

rfm.rename(columns={
    'InvoiceDate': 'Recency',
    'Invoice': 'Frequency',
    'TotalPrice': 'Monetary'
}, inplace=True)

print(f"RFM Matrix entries: {len(rfm)}")

RFM Matrix entries: 4933


## 4. Define CLV Target (Observation)
The target is the actual revenue from each customer in the 6 months *after* the split.

In [5]:
target = df_obs.groupby('Customer ID')['TotalPrice'].sum()
rfm['CLV_Target'] = target.reindex(rfm.index, fill_value=0)

proc_path = os.path.join(PROCESSED_DIR, 'rfm_clv_data.csv')
rfm.to_csv(proc_path)
print(f"Processed data saved to {proc_path}")
rfm.head()

Processed data saved to /home/ashutosh/Desktop/NO-AI-USE/ml_projects/03_customer_segmentation_clv/data/processed/rfm_clv_data.csv


,Recency,Frequency,Monetary,CLV_Target
Customer ID,,,,
12346.0,133,12,77556.46,0.00
12347.0,54,4,3146.75,2486.57
12348.0,56,4,1709.40,310.00
12349.0,215,3,2671.14,1757.55
12350.0,118,1,334.40,0.00
